## 1. 根据氨基酸序列表找到对应的核苷酸序列

In [2]:
import pandas as pd

CDC_H1N1 = pd.read_excel('../../data/raw/data4model(CDC-H1N1).xlsx')
CDC_H3N2 = pd.read_excel('../../data/raw/data4model(CDC-H3N2).xlsx')

In [3]:
def read_fasta(file_path):
    sequence_names = []
    sequences = []
    current_sequence = []

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                # 如果当前序列不为空，保存当前序列
                if current_sequence:
                    sequences.append(''.join(current_sequence))
                    current_sequence = []
                # 保存序列名称
                sequence_names.append(line[1:])
            else:
                # 将序列数据添加到当前序列
                current_sequence.append(line)

        # 不要忘记保存最后一个序列
        if current_sequence:
            sequences.append(''.join(current_sequence))

    return sequence_names, sequences

def seq2DF(names, seqs):
    Meta = pd.DataFrame([s.split('|') for s in names])
    Meta['Sequence'] = seqs
    return Meta

In [4]:
sequence_names, sequences = read_fasta('../../data/processed/H1_DNA.fasta')
H1_DNA = seq2DF(sequence_names, sequences)
H1_DNA.columns = ['virusName','EPI_Isl_ID','Passage','lineage','collectionDate','EPI_ID','Sequence']
sequence_names, sequences = read_fasta('../../data/processed/H3_DNA.fasta')
H3_DNA = seq2DF(sequence_names, sequences)
H3_DNA.columns = ['virusName','EPI_Isl_ID','Passage','lineage','collectionDate','EPI_ID','Sequence']

In [5]:
sequence_names, sequences = read_fasta('../../data/processed/H1_AA.fasta')
H1_AA = seq2DF(sequence_names, sequences)
H1_AA = H1_AA.iloc[:,[0,1,3,6,14,16]].copy()
H1_AA.columns = ['virusName','EPI_Isl_ID','Passage','collectionDate','EPI_ID','Sequence']
sequence_names, sequences = read_fasta('../../data/processed/H3_AA.fasta')
H3_AA = seq2DF(sequence_names, sequences)
H3_AA = H3_AA.iloc[:,[0,1,3,6,14,17]].copy()
H3_AA.columns = ['virusName','EPI_Isl_ID','Passage','collectionDate','EPI_ID','Sequence']

In [6]:
H1N1_integrated = CDC_H1N1[['serumName','serumPassage', 'serumPassCat', 'virusName', 
                              'virusPassage','virusPassCat', 'serumIslID', 'virusIslID',
                              'serumHA', 'virusHA', 'HI_Dist']].copy()
H3N2_integrated = CDC_H3N2[['serumName','serumPassage', 'serumPassCat', 'virusName', 
                              'virusPassage','virusPassCat', 'serumIslID', 'virusIslID',
                              'serumHA', 'virusHA', 'HI_Dist']].copy()
H1N1_integrated['serumType'] = 'H1N1'
H3N2_integrated['serumType'] = 'H3N2'
H1N1_integrated.rename(columns={'serumHA':'serumHA_AA','virusHA':'virusHA_AA'},inplace=True)
H3N2_integrated.rename(columns={'serumHA':'serumHA_AA','virusHA':'virusHA_AA'},inplace=True)

In [7]:
filt1 = pd.merge(H1N1_integrated,H1_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['serumIslID','serumHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'serum_HA_EPI_ID'})
filt2 = pd.merge(filt1,H1_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['virusIslID','virusHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'virus_HA_EPI_ID'})

filt3 = pd.merge(filt2,H1_DNA[['EPI_ID','Sequence']],
                 left_on=['serum_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'serumHA_DNA'})
filt4 = pd.merge(filt3,H1_DNA[['EPI_ID','Sequence']],
                 left_on=['virus_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'virusHA_DNA'})
H1N1_final = filt4.copy()

filt1 = pd.merge(H3N2_integrated,H3_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['serumIslID','serumHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'serum_HA_EPI_ID'})
filt2 = pd.merge(filt1,H3_AA[['EPI_Isl_ID','Sequence','EPI_ID']],
                 left_on=['virusIslID','virusHA_AA'],
                 right_on=['EPI_Isl_ID','Sequence'],how='left').drop(columns=['EPI_Isl_ID','Sequence']) \
                 .rename(columns={'EPI_ID':'virus_HA_EPI_ID'})

filt3 = pd.merge(filt2,H3_DNA[['EPI_ID','Sequence']],
                 left_on=['serum_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'serumHA_DNA'})
filt4 = pd.merge(filt3,H3_DNA[['EPI_ID','Sequence']],
                 left_on=['virus_HA_EPI_ID'],
                 right_on=['EPI_ID'],how='left').drop(columns=['EPI_ID']) \
                 .rename(columns={'Sequence':'virusHA_DNA'})
H3N2_final = filt4.copy()

In [8]:
Final_df = pd.concat([H1N1_final, H3N2_final], axis=0).reset_index(drop=True)

In [10]:
Final_df.to_csv('../../data/processed/0.1_Final_df-CDC.csv',index=False)